# Classification Small — Full Pipeline with GReaT (GPU)
**IRIT / Université Toulouse Capitole — Counterfactual Data Augmentation**

Run each cell in order. Step 1 must be done manually (upload zip).

## Step 1 — Upload your Implementation folder

1. On your Windows machine, zip the entire `Implementation/` folder
2. Upload `Implementation.zip` using the file panel on the left (📁 icon)
3. Then run the cell below to unzip it

In [11]:
import os
os.chdir('/projects/sig/usman.ashfaq/Implementation')
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

Working directory: /projects/sig/usman.ashfaq/Implementation
Files: ['Classification', '.ipynb_checkpoints', 'outputs', 'utils', 'requirements.txt', 'clf_small.py', 'trainer_great', 'ForestDiffusion', 'clf_small_colab.ipynb']


## Step 2 — Check GPU

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU')

CUDA available: True
GPU: NVIDIA L40S
Memory: 47.7 GB


## Step 3 — Install all required packages

In [2]:
%%capture
!pip install sdv dice-ml shap imbalanced-learn xgboost be-great smogn forestdiffusion
!pip install "numpy==2.2.6"   # pin numpy — numba (used by SHAP) needs <= 2.3
print('All packages installed')

In [3]:
# Verify all critical imports
checks = [
    ('from ForestDiffusion import ForestDiffusionModel', 'TabDDPM (ForestDiffusion)'),
    ('from be_great import GReaT',                       'GReaT'),
    ('import dice_ml',                                   'DiCE'),
    ('import sdv',                                       'CTGAN/TVAE'),
    ('import shap',                                      'SHAP'),
    ('from imblearn.over_sampling import SMOTENC',       'SMOTE-NC'),
    ('import xgboost',                                   'XGBoost'),
    ('import torch; assert torch.cuda.is_available()',   'GPU/CUDA'),
]
all_ok = True
for imp, name in checks:
    try:
        exec(imp)
        print(f'  OK   {name}')
    except Exception as e:
        print(f'  FAIL {name}: {e}')
        all_ok = False

print()
print('Ready to run!' if all_ok else 'Fix the FAIL items above before continuing.')

  OK   TabDDPM (ForestDiffusion)


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  OK   GReaT
  OK   DiCE
  OK   CTGAN/TVAE
  OK   SHAP
  OK   SMOTE-NC
  OK   XGBoost
  OK   GPU/CUDA

Ready to run!


## Step 4 — Confirm GReaT is enabled in config

In [12]:
import sys
sys.path.insert(0, '/projects/sig/usman.ashfaq/Implementation')  # ← remove /utils

from utils.config import METHODS, GREAT_EPOCHS, SEEDS, CLF_DATASETS  # ← remove DATASETS

print('Methods  :', METHODS)
print('GReaT epochs:', GREAT_EPOCHS)
print('Seeds    :', SEEDS)
print('Small datasets:', CLF_DATASETS['small'])

assert 'great' in METHODS, "Add 'great' to METHODS in utils/config.py"
assert GREAT_EPOCHS['small'] >= 25, "GREAT_EPOCHS should be 25 for small datasets"
print('\nConfig OK - GReaT enabled with', GREAT_EPOCHS['small'], 'epochs for small datasets')

Methods  : ['baseline', 'smote_nc', 'ctgan', 'tvae', 'tabddpm', 'great', 'dice']
GReaT epochs: {'small': 25, 'medium': 75, 'large': 50}
Seeds    : [42, 123, 456, 789, 999]
Small datasets: ['heart_c', 'saheart', 'australian', 'german']

Config OK - GReaT enabled with 25 epochs for small datasets


## Step 5 — Quick smoke test (1 dataset, 1 seed) — run this first!

In [13]:
# Temporarily override for smoke test only
import utils.config as cfg
_orig_datasets = cfg.CLF_DATASETS.copy()
_orig_seeds = cfg.SEEDS[:]

cfg.CLF_DATASETS['small'] = ['german']   # safest small dataset
cfg.SEEDS = [42]                          # one seed only

print('Smoke test: german dataset, seed 42 only')
print('This should take ~15-20 minutes with GPU (GReaT dominates)')

Smoke test: german dataset, seed 42 only
This should take ~15-20 minutes with GPU (GReaT dominates)


In [14]:
# Run the smoke test
%run clf_small.py


  CLASSIFICATION - SMALL DATASETS  |  Seeds: [42]

  Dataset: HEART_C  |  Task: classification  |  Size: small

  Dataset : heart_c
  Samples : 303  |  Features: 13
  binary=3  cat=5  cont=5
  Classes : 2  |  Missing: 0


--------------------------------------------------
    Seed 42
--------------------------------------------------

  -> [heart_c] seed=42  method=baseline
    Computing SHAP (RF)...
    Computing SHAP (XGB)...

  -> [heart_c] seed=42  method=smote_nc
  Running: SMOTE_NC
    Computing SHAP (RF)...
    Computing SHAP (XGB)...

  -> [heart_c] seed=42  method=ctgan
  Running: CTGAN
    Computing SHAP (RF)...
    Computing SHAP (XGB)...

  -> [heart_c] seed=42  method=tvae
  Running: TVAE
    Computing SHAP (RF)...
    Computing SHAP (XGB)...

  -> [heart_c] seed=42  method=tabddpm
  Running: TABDDPM
  [TabDDPM] ForestDiffusion OK — generated 18 rows
    Computing SHAP (RF)...
    Computing SHAP (XGB)...

  -> [heart_c] seed=42  method=great
  Running: GREAT


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1608.09it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.29s/it]


{'train_runtime': '16.89', 'train_samples_per_second': '313.7', 'train_steps_per_second': '10.36', 'train_loss': '1.097', 'epoch': '25'}


INFO:classification_small:  [great] Generated 18 synthetic rows in 36.1s
INFO:classification_small:  [great] RF  F1=0.7651  AUC=0.8739
INFO:classification_small:  [great] XGB F1=0.7796  AUC=0.8634


    Computing SHAP (RF)...
    Computing SHAP (XGB)...

  -> [heart_c] seed=42  method=dice
  Running: DICE


... (76 outputs left out)

INFO:classification_small:  [dice] RF  F1=0.6738  AUC=0.8128
INFO:classification_small:  [dice] XGB F1=0.7137  AUC=0.7744


    Computing SHAP (RF)...
    Computing SHAP (XGB)...


INFO:classification_small:  Dataset german complete. Outputs -> /projects/sig/usman.ashfaq/Implementation/outputs/classification/small/german
INFO:classification_small:All outputs saved under: /projects/sig/usman.ashfaq/Implementation/outputs/classification/small



  ALL SMALL CLASSIFICATION DATASETS COMPLETE


In [15]:
# Check smoke test outputs
import os, glob
output_base = '/content/Implementation/outputs/classification/small/german'
all_files = glob.glob(output_base + '/**/*', recursive=True)
print(f'Files created: {len(all_files)}')
for f in sorted(all_files)[:30]:
    print(' ', f.replace(output_base, ''))

# Check GReaT specifically produced output
great_files = [f for f in all_files if 'great' in f]
print(f'\nGReaT output files: {len(great_files)}')
if great_files:
    print('GReaT SUCCESS — proceed to full run')
else:
    print('GReaT produced no files — check errors above')

Files created: 0

GReaT output files: 0
GReaT produced no files — check errors above


## Step 6 — Full run (all 4 datasets, all 5 seeds)

Only run this after the smoke test passes. Expected time on GPU: **2–4 hours**.

In [16]:
# Restore full config
import utils.config as cfg
cfg.CLF_DATASETS['small'] = ['heart_c', 'saheart', 'australian', 'german']
cfg.SEEDS = [42, 123, 456, 789, 999]

print('Full run config:')
print('  Datasets:', cfg.CLF_DATASETS['small'])
print('  Seeds   :', cfg.SEEDS)
print('  Methods :', cfg.METHODS)

Full run config:
  Datasets: ['heart_c', 'saheart', 'australian', 'german']
  Seeds   : [42, 123, 456, 789, 999]
  Methods : ['baseline', 'smote_nc', 'ctgan', 'tvae', 'tabddpm', 'great', 'dice']


## Step 7 — Download results

In [ ]:
# Zip and download the outputs
import shutil, os

output_dir = '/content/Implementation/outputs/classification/small'
zip_path   = '/content/clf_small_results'

shutil.make_archive(zip_path, 'zip', output_dir)
print(f'Zipped: {zip_path}.zip')
size = os.path.getsize(f'{zip_path}.zip') / 1e6
print(f'Size: {size:.1f} MB')

In [ ]:
# Run full experiment — this is the main run
%run clf_small.py

In [ ]:
# Download to your computer
from google.colab import files
files.download('/content/clf_small_results.zip')

## Notes

**Expected GReaT behaviour on GPU:**
- Training: ~2-5 min per dataset (25 epochs on T4 GPU)
- Sampling: ~1-2 min per dataset
- Total GReaT contribution: ~15-20 min per seed across 4 datasets

**If GReaT still fails to sample** (rare on GPU with 25 epochs):
- It will fall back to `guided_sampling=True` automatically
- If both fail, it logs a warning and skips cleanly
- This is acceptable — document in paper's limitations

**After downloading results:**
- Extract `clf_small_results.zip` into `Implementation/outputs/classification/small/`
- The Windows run results (all methods except GReaT) are already there
- The Colab results (including GReaT) will merge with them